# Lung Histopathology: Dataset Setup and Integrity Verification

**Study:** Stain-Robust Explainable Deep Feature Fusion Framework for Lung Cancer Histopathology Subtype Classification

This notebook verifies the original LC25000 lung-only input archive, immutable split assignments, reconstructed source-image groups, and basic image integrity before opening the experiment notebooks. It is intended for Google Colab with the original saved project data on Google Drive.

**Scope:** Data availability, provenance, split checks, archive inventory, and a small training/validation image-decoding smoke check. This notebook never trains a model or uses held-out test image pixels. The experiment notebooks contain their own full train/validation restoration and training protocols.

**Inputs that must already exist:** The original `lung/archives/LC25000.zip`, `lung/splits/manifest_candidate.csv`, `lung/splits/class_mapping.json`, `lung/metadata/lc25000_image_groups.csv`, and `lung/results/audit.json`. These exact saved records are the basis of the completed experiments. If a file is missing, restore the original verified file rather than generating a new split.

**Execution:** Run the cells sequentially. The checksum of the approximately 1.8 GiB archive may take time when read from Drive. The notebook does not write to the original Drive folders.

## 1. Mount Google Drive and locate the original lung project inputs

The only directory created by this notebook is a temporary local workspace under `/content`. Existing Drive folders are never created, renamed, moved, or overwritten.

In [1]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from datetime import datetime, timezone
import hashlib
import io
import json
import os
import platform
import sys
import zipfile

import pandas as pd

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Run this notebook in Google Colab.') from exc

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/ICTIU_Final_Projects/lung')
WORK = Path('/content/ictiu_data/lung_setup')
WORK.mkdir(parents=True, exist_ok=True)

SOURCES = {
    'archive': PROJECT / 'archives/LC25000.zip',
    'manifest': PROJECT / 'splits/manifest_candidate.csv',
    'mapping': PROJECT / 'splits/class_mapping.json',
    'groups': PROJECT / 'metadata/lc25000_image_groups.csv',
    'audit': PROJECT / 'results/audit.json',
}

if not PROJECT.is_dir():
    raise FileNotFoundError(
        f'Original Drive project folder is missing: {PROJECT}. '
        'Use the original account or restore the verified input files first.'
    )

for role, path in SOURCES.items():
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f'Missing original {role}: {path}. Restore this exact saved input. '
            'Do not regenerate split assignments.'
        )
    print(f'FOUND: {path.relative_to(PROJECT)}')

print('Original lung data sources located.')
print('Local temporary workspace:', WORK)
print('Original Drive files were not modified.')

Mounted at /content/drive
FOUND: archives/LC25000.zip
FOUND: splits/manifest_candidate.csv
FOUND: splits/class_mapping.json
FOUND: metadata/lc25000_image_groups.csv
FOUND: results/audit.json
Original lung data sources located.
Local temporary workspace: /content/ictiu_data/lung_setup
Original Drive files were not modified.


## 2. Verify original source-file fingerprints

Compare the files against SHA-256 fingerprints recorded during the completed lung experiments. The archive checksum checks the stored compressed data, not biological independence.

In [2]:
EXPECTED_SHA256 = {
    'archive': '382c98315435cbbf559be5da94cc0b62f696434e30e15feefebe9c892de01728',
    'manifest': 'd931c9b11d03716c3c426f251bc75eaa2051002acca99d3ba66d59d964285dea',
    'mapping': '4d14782118e3a438c13b209d7a594f2f9676d09cec25125b599d8c6d6c26a6a8',
    'groups': '2ff9e992f759f265dfc9aa2363de0dfcc8e6559884d8bb02a4c925214fdf7e23',
    'audit': 'cd9aecfb8bfe64ca956b205e333ce9c9697dbfa88d4c46dc89c3ac088115fac0',
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for chunk in iter(lambda: source.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

verified_hashes = {}
for role, expected in EXPECTED_SHA256.items():
    actual = sha256_file(SOURCES[role])
    if actual != expected:
        raise ValueError(f'{role} SHA-256 mismatch: {actual}; expected {expected}')
    verified_hashes[role] = actual
    print(f'PASS {role:8s} SHA-256: {actual}')

print('PASS: The five original source files match the recorded fingerprints.')

PASS archive  SHA-256: 382c98315435cbbf559be5da94cc0b62f696434e30e15feefebe9c892de01728
PASS manifest SHA-256: d931c9b11d03716c3c426f251bc75eaa2051002acca99d3ba66d59d964285dea
PASS mapping  SHA-256: 4d14782118e3a438c13b209d7a594f2f9676d09cec25125b599d8c6d6c26a6a8
PASS groups   SHA-256: 2ff9e992f759f265dfc9aa2363de0dfcc8e6559884d8bb02a4c925214fdf7e23
PASS audit    SHA-256: cd9aecfb8bfe64ca956b205e333ce9c9697dbfa88d4c46dc89c3ac088115fac0
PASS: The five original source files match the recorded fingerprints.


## 3. Validate the immutable class mapping and source-group split

A reconstructed source-image group is not a patient, slide, or hospital identifier. Verify group separation at the level available in LC25000-clean; do not describe this split as patient-independent.

In [3]:
EXPECTED_CLASSES = {'lung_aca': 0, 'lung_n': 1, 'lung_scc': 2}
EXPECTED_IMAGES = {'train': 10394, 'validation': 2273, 'test': 2333}
EXPECTED_GROUPS = {'train': 514, 'validation': 113, 'test': 113}
REQUIRED_COLUMNS = {'relative_path', 'class_name', 'group_id', 'split', 'sha256', 'class_id'}

with SOURCES['mapping'].open(encoding='utf-8-sig') as stream:
    mapping = json.load(stream)
with SOURCES['audit'].open(encoding='utf-8-sig') as stream:
    audit = json.load(stream)
manifest = pd.read_csv(SOURCES['manifest'], dtype=str, keep_default_na=False)

if mapping != EXPECTED_CLASSES:
    raise ValueError(f'Class mapping differs from the recorded order: {mapping}')
if not REQUIRED_COLUMNS.issubset(manifest.columns):
    raise ValueError(f'Missing manifest columns: {sorted(REQUIRED_COLUMNS - set(manifest.columns))}')
if len(manifest) != 15000:
    raise ValueError(f'Expected 15,000 manifest rows, found {len(manifest)}.')
if not manifest['split'].isin(EXPECTED_IMAGES).all():
    raise ValueError('Manifest contains unexpected split labels.')
if not manifest['class_name'].isin(EXPECTED_CLASSES).all():
    raise ValueError('Manifest contains unexpected class labels.')
if not manifest['sha256'].str.fullmatch(r'[0-9a-fA-F]{64}').all():
    raise ValueError('Invalid saved image-hash column.')
if (manifest[['relative_path', 'group_id', 'class_name']] == '').any().any():
    raise ValueError('A manifest identity field is empty.')
if manifest['relative_path'].duplicated().any():
    raise ValueError('Duplicate saved image paths.')
if not all(str(EXPECTED_CLASSES[n]) == k for n, k in zip(manifest['class_name'], manifest['class_id'])):
    raise ValueError('Class names and numeric IDs are inconsistent.')

image_counts = manifest['split'].value_counts().to_dict()
group_counts = manifest.groupby('split')['group_id'].nunique().to_dict()
if image_counts != EXPECTED_IMAGES or group_counts != EXPECTED_GROUPS:
    raise ValueError(f'Unexpected saved split counts: {image_counts}; groups: {group_counts}')
if manifest['group_id'].nunique() != 740:
    raise ValueError('Expected 740 reconstructed source-image groups.')
if manifest.groupby('group_id')['split'].nunique().max() != 1:
    raise ValueError('Source-image group crosses split boundaries.')
if manifest.groupby('group_id')['class_name'].nunique().max() != 1:
    raise ValueError('Source-image group crosses class boundaries.')
if manifest.groupby('sha256')['split'].nunique().max() != 1:
    raise ValueError('Exact duplicate image hashes cross splits.')
if manifest['class_name'].value_counts().to_dict() != {c:5000 for c in EXPECTED_CLASSES}:
    raise ValueError('Original lung-only classes are not 5,000 images each.')
if audit.get('manifest_sha256') != verified_hashes['manifest']:
    raise ValueError('The saved audit does not match the saved manifest.')
if audit.get('status') != 'PASS_candidate_split_checks' or audit.get('group_overlap') != 0:
    raise ValueError('The original preparation audit is inconsistent.')

print('PASS: Original class order:', EXPECTED_CLASSES)
print('PASS: Lung-only images per split:', image_counts)
print('PASS: Reconstructed source groups per split:', group_counts)
print('PASS: Group and exact-byte-hash split overlap checks.')
print('No new split was generated. No TEST image pixels were accessed.')

PASS: Original class order: {'lung_aca': 0, 'lung_n': 1, 'lung_scc': 2}
PASS: Lung-only images per split: {'train': 10394, 'test': 2333, 'validation': 2273}
PASS: Reconstructed source groups per split: {'test': 113, 'train': 514, 'validation': 113}
PASS: Group and exact-byte-hash split overlap checks.
No new split was generated. No TEST image pixels were accessed.


## 4. Inspect ZIP member names without extracting any images

Resolve every training and validation row to one unique archive member using the class and basename, matching the original experiment notebooks' archive lookup. This stage reads the archive index only. Test image files are not opened or extracted.

In [4]:
from collections import defaultdict

train_val = manifest.loc[manifest['split'].isin(['train', 'validation'])].copy()
if len(train_val) != 12667:
    raise ValueError('Unexpected training/validation row count.')

member_index = defaultdict(list)
with zipfile.ZipFile(SOURCES['archive'], mode='r') as archive:
    for info in archive.infolist():
        path = PurePosixPath(info.filename.replace('\\', '/'))
        if info.is_dir() or '__MACOSX' in path.parts or path.name.startswith('._'):
            continue
        if path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
            continue
        classes = [p for p in path.parts if p in EXPECTED_CLASSES]
        if len(classes) == 1:
            member_index[(classes[0], path.name)].append(info.filename)

    selected_members = []
    for row in train_val.itertuples(index=False):
        basename = PurePosixPath(row.relative_path.replace('\\', '/')).name
        matches = member_index.get((row.class_name, basename), [])
        if len(matches) != 1:
            raise ValueError(f'Expected one ZIP match for {row.class_name}/{basename}; found {len(matches)}.')
        name = matches[0]
        parts = PurePosixPath(name).parts
        if '..' in parts or PurePosixPath(name).is_absolute():
            raise ValueError(f'Unsafe archive path: {name}')
        selected_members.append(name)

if len(set(selected_members)) != len(train_val):
    raise ValueError('At least two manifest rows resolve to one archive image.')

print(f'PASS: {len(selected_members):,} unique training/validation ZIP image members resolved.')
print('No archive images extracted; TEST image pixels not accessed.')

PASS: 12,667 unique training/validation ZIP image members resolved.
No archive images extracted; TEST image pixels not accessed.


## 5. Decode a deterministic training/validation-only sample

Read only nine selected training/validation samples from the original ZIP. Verify saved SHA-256 values and image decoding, without writing medical images to Drive. This is a smoke check, not a substitute for the full image checks in H1/H2.

In [5]:
from PIL import Image

smoke_rows = (
    train_val.sort_values(['class_name', 'relative_path'])
    .groupby('class_name', sort=True)
    .head(3)
)
assert len(smoke_rows) == 9
checked = []
with zipfile.ZipFile(SOURCES['archive'], mode='r') as archive:
    for row in smoke_rows.itertuples(index=False):
        basename = PurePosixPath(row.relative_path.replace('\\', '/')).name
        entry = member_index[(row.class_name, basename)][0]
        raw = archive.read(entry)
        digest = hashlib.sha256(raw).hexdigest()
        if digest != row.sha256.lower():
            raise ValueError(f'Image hash mismatch for {row.class_name}/{basename}')
        with Image.open(io.BytesIO(raw)) as image:
            image.verify()
        checked.append(f'{row.class_name}/{basename}')

print('PASS: SHA-256 and image decoding for 9 deterministic training/validation images.')
print('Checked classes:', {c: 3 for c in EXPECTED_CLASSES})
print('No TEST image pixels read; no medical images saved.')

PASS: SHA-256 and image decoding for 9 deterministic training/validation images.
Checked classes: {'lung_aca': 3, 'lung_n': 3, 'lung_scc': 3}
No TEST image pixels read; no medical images saved.


## 6. Record a read-only environment summary

The exact historical training environment is documented separately. This cell prints the current runtime for comparison and saves a local JSON summary in `/content`, not to a permanent experiment results directory.

In [6]:
from importlib import metadata

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

runtime = {
    'captured_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'packages': {pkg: package_version(pkg) for pkg in [
        'tensorflow', 'keras', 'numpy', 'pandas', 'scikit-learn',
        'matplotlib', 'pillow', 'tqdm', 'opencv-python-headless', 'opencv-python'
    ]},
    'archive_sha256': verified_hashes['archive'],
    'manifest_sha256': verified_hashes['manifest'],
    'mapping_sha256': verified_hashes['mapping'],
    'grouping_unit': 'reconstructed source-image groups; not patient IDs',
    'source_image_groups_verified': 740,
    'counts_verified': EXPECTED_IMAGES,
    'test_image_pixels_read': False,
    'new_split_generated': False,
    'source_data_modified': False,
}
summary_path = WORK / 'lung_setup_verification.json'
summary_path.write_text(json.dumps(runtime, indent=2), encoding='utf-8')
print('PASS: Lung preparation inputs verified; original Drive data unchanged.')
print('Runtime summary saved locally:', summary_path)
print('This setup notebook does not restore all images or rerun experiments.')

PASS: Lung preparation inputs verified; original Drive data unchanged.
Runtime summary saved locally: /content/ictiu_data/lung_setup/lung_setup_verification.json
This setup notebook does not restore all images or rerun experiments.


## Use with the existing experiment archive

The executed H1, H2, H3, H4, Grad-CAM, and final TEST notebooks are historical experiment records. Their saved output cells document the completed work. H1/H2 contain their own train/validation image restoration procedures. H3 uses the saved H1/H2 feature extractors, and H4 contains separate stain-processing ablations.

The selected H3 held-out TEST was already evaluated once and must **not** be rerun for GitHub documentation or report editing. Original model checkpoints, inference evidence, and split files remain in the project owner's Google Drive, outside this Git repository. For any future independently authorized training reproduction, use fresh output directories and preserve the original manifest assignments and checkpoint hashes.